In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from ift6758.llm.prompts import PROFILES,OG_TACHE_1
from ift6758.llm.rag import load_doc, extract_section, split_doc, load_embedder, embed, retrieve, format_context,DOCS_DIR
from ift6758.llm.qwen_log import create_log,log_call
from ift6758.llm.qwen_utils import load_qwen,build_messages,generate
from ift6758.llm.test_both_code import check_download, check_cache_no_request, check_failed_request,check_wrong_cache, run_tests
from ift6758.llm.get_better import extract_text,build_refine_prompt,load_function,format_test_report
import requests
import json
from pathlib import Path
from tempfile import TemporaryDirectory

In [3]:
filename = "nhl-api-reference.md"
url = "https://raw.githubusercontent.com/Zmalski/NHL-API-Reference/main/README.md"

text = load_doc(url, filename)
section = extract_section(text,"# NHL Web API Documentation","# NHL Stats API Documentation")
chunks = split_doc(section)
embedder = load_embedder()
vectors = embed(chunks,embedder)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [4]:
model, tokenizer, device = load_qwen()
log_path = create_log()

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

In [6]:
profile_name = "gen_code"
question = OG_TACHE_1
version_prompt = "v0"
p = PROFILES[profile_name]
#confi rag.py
query = p["retrieval_query"] or question
results = retrieve(query,chunks,vectors,embedder,k=p["k"])
context = format_context(results,source=filename)
# config qwen
messages = build_messages(question,context,system=p["system"])
llm_result = generate(model,tokenizer,messages,max_new_tokens=p["max_new_tokens"],do_sample=p["do_sample"])
# config log
log_call(log_path,messages,llm_result,extra={"profile":profile_name,"profile_version":p["version"],"retrieved":results, "max_new_tokens": p["max_new_tokens"],"do_sample": p["do_sample"], "context_format":"v5", "task_version":version_prompt})
print(llm_result["answer"])
print(f"\n{llm_result['n_token_in']} tokens en entrée, {llm_result['n_token_out']} générés, {llm_result['duration_s']:.1f} s")

```python
import requests
import json
from pathlib import Path

def fetch_game_llm(game_id, cache_dir):
    """
    Fetch play-by-play data for a given game ID from the NHL API with caching.
    
    Args:
        game_id (int): The ID of the game to fetch data for.
        cache_dir (str): Directory path where cached JSON files will be stored.
    
    Returns:
        dict: The full game dictionary containing 'id' (int) and 'plays' (list).
    
    Raises:
        requests.exceptions.RequestException: If the API request fails.
        ValueError: If the response data does not contain valid 'id' or 'plays'.
    """
    url = f"https://api-web.nhle.com/v1/gamecenter/{game_id}/play-by-play"
    cache_path = Path(cache_dir) / f"{game_id}.json"
    
    # Create cache directory if it doesn't exist
    cache_path.parent.mkdir(parents=True, exist_ok=True)
    
    # Check if cached data exists
    if cache_path.exists():
        with open(cache_path, 'r', encoding='utf-8') as f:
           

In [8]:
GAME_ID = 2022030411
raw_path = Path(DOCS_DIR.parent / "raw") / f"{GAME_ID}.json"
print("Cached game:", raw_path.resolve())
if not raw_path.exists():
    response = requests.get(f"https://api-web.nhle.com/v1/gamecenter/{GAME_ID}/play-by-play", timeout=30)
    response.raise_for_status()
    downloaded = response.json()
    assert downloaded["id"] == GAME_ID and isinstance(downloaded["plays"], list)
    raw_path.parent.mkdir(parents=True, exist_ok=True)
    raw_path.write_text(response.text, encoding="utf-8")
game = json.loads(raw_path.read_text(encoding="utf-8")) # Convertie de json vers objet python
assert game["id"] == GAME_ID and isinstance(game["plays"], list)
print(game["awayTeam"]["abbrev"], "at", game["homeTeam"]["abbrev"], game["gameDate"])

Cached game: /Users/jcforget/Maitrise/Cours/IFT 6758/Projet/Github/NHL-data-analysis/ift6758/data/raw/2022030411.json
FLA at VGK 2023-06-03


Extraction contexte pour génération nouveau prompt

In [13]:
from ift6758.llm.get_better import extract_text,build_refine_prompt,load_function,format_test_report
code = extract_text(answer=llm_result["answer"],tag="code")
print(code) # Pour voir si extraction bonne
fn = load_function(code,name = "fetch_game_llm")
test_results = run_tests(fetch_fn=fn,game_id=GAME_ID,game=game)
report = format_test_report(test_results)
new_context = build_refine_prompt(question,code,context,report)
print(f"\nNew context: {new_context}\n")
profile_name = "refine"
p = PROFILES[profile_name]

KeyboardInterrupt: 

Génération nouveau Prompt

In [7]:
prompt_it = 1
message = build_messages(question,new_context,system=p["system"])
answer = generate(model,tokenizer,messages,max_new_tokens=p["max_new_tokens"],do_sample=p["do_sample"])
new_question = extract_text(answer=answer["answer"],type="new_prompt")
log_call(log_path,messages,llm_result,extra={"profile":profile_name,"profile_version":p["version"],"retrieved":results, "max_new_tokens": p["max_new_tokens"],"do_sample": p["do_sample"], "context_format":"v5", "task_version":version_prompt})
print(llm_result["answer"])
print(f"\n{llm_result['n_token_in']} tokens en entrée, {llm_result['n_token_out']} générés, {llm_result['duration_s']:.1f} s")

NameError: name 'new_context' is not defined

Extraction pour nouveau code

In [ ]:
profile_name = "gen_code"
version_prompt = f"v{prompt_it}"
p = PROFILES[profile_name]
#confi rag.py
query = p["retrieval_query"] or new_question
results = retrieve(query,chunks,vectors,embedder,k=p["k"])
context = format_context(results,source=filename)

Génération nouveau code

In [ ]:
message = build_messages(question,new_context,system=p["system"])
new_question = generate(model,tokenizer,messages,max_new_tokens=p["max_new_tokens"],do_sample=p["do_sample"])
log_call(log_path,messages,llm_result,extra={"profile":profile_name,"profile_version":p["version"],"retrieved":results, "max_new_tokens": p["max_new_tokens"],"do_sample": p["do_sample"], "context_format":"v5", "task_version":version_prompt})
print(llm_result["answer"])
print(f"\n{llm_result['n_token_in']} tokens en entrée, {llm_result['n_token_out']} générés, {llm_result['duration_s']:.1f} s")